# TD3: Twin Delayed DDP
TD3 （Twin Delayed DDPG）はActor-Critic系強化学習手法であるDDPGの改良手法です。
基本的な流れはDDPGとほぼ同じですが、Double DQN論文が指摘したDQNでのQ関数の過大評価がActor-Criticでも生じることを示し、学習安定化のために下記の３つのテクニックを提案しました。

1. Clipped Double Q learning
2. Target Policy Smoothing
3. Delayed Policy Update



### リファレンス
- [Ｐｙｔｈｏｎで学ぶ強化学習](https://www.amazon.co.jp/dp/B082HNNGQG/)
- [TD3の解説・実装（強化学習）](https://horomary.hatenablog.com/entry/2020/07/01/001414)

### code
- [icoxfog417/baby-steps-of-rl-ja](https://github.com/icoxfog417/baby-steps-of-rl-ja)
- [horoiwa/deep_reinforcement_learning_gallery](https://github.com/horoiwa/deep_reinforcement_learning_gallery/tree/master)

In [1]:
import torch

from code.pendulum_observer import PendulumObserver
from code.td3_trainer import TD3Trainer
from code.td3_agent import TD3Agent


In [2]:
device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
print('device: ', device)
Training = False

device:  mps


### Experiment
updateメソッドの中の`target_values = rewards + self.gamma * masks * min_q_next`masksの値と、
計算結果について確認する。

In [12]:
import numpy as np

dones = np.array([False, False, False, False])
values = np.array([1, 1, 1, 1])
print("values * dones: ", values * dones)
print("values * (1 - dones): ", values * (1 - dones))

masks = []
for d in dones:
    mask = float(not d)
    masks.append(mask)
print("valus * masks: ", values * masks)

values * dones:  [0 0 0 0]
values * (1 - dones):  [1 1 1 1]
valus * masks:  [1. 1. 1. 1.]


<ChatGPTによる説明>  
dones は環境から返される終端判定(エピソードが終了したかどうか)の情報ですが、コードの変更前は dones をそのまま (1 - dones) として割り当てていました。これは dones が0/1形式で「終了で1」「未終了で0」という単純なブーリアンフラグとして処理できることが前提となります。しかし、環境や実装によっては dones が浮動小数点や、異なる形式で返される場合があり、想定どおりにターミナル状態を反映していなかった可能性があります。

また、一部のGym環境や実装では、doneではなく、最後のステップでもう少し複雑な情報が返されることもあり得ます。そのため (1 - dones) による単純なターミナル状態判別が期待どおり機能せず、ターゲット値計算が常にずれ続けて学習が進まない、あるいは不安定になることがあります。

一方、変更後のコードでは、mask = float(not done) という明確なマスク値がメモリに格納され、それを用いることで、以下のように状態遷移が終端であるか否かを明示的に管理しています。

### Training

In [ ]:
if Training:
    observer = PendulumObserver()
    model_path = "models/td3_agent.pth"
    trainer = TD3Trainer(model_path=model_path, device=device)
    agent = trainer.train(env=observer)

### Play

In [6]:
observer = PendulumObserver(play=True)
actor_model_path = "models/td3_agent_best_actor.pth"
critic_model_path = "models/td3_agent_best_critic.pth"
agent = TD3Agent.load(device=device, state_dim=3,action_dim=observer.action_space, actor_model_path=actor_model_path, critic_model_path=critic_model_path)
agent.play(observer, render=False)

episode 0: -245.82572809985538
episode 1: -462.71666040996604
episode 2: -1558.3388786066714
episode 3: -0.7267040839378618
episode 4: -247.78005422101648
episode 5: -1600.787483749667
episode 6: -250.67529184580187
episode 7: -1525.961353750309
episode 8: -1.3504002150511236
episode 9: -0.8605716292417411
